# Assignment 02 - PIGEONS LOVE PIAZZA DUOMO
**Michelle Nuccio**

I follow Tutorial 02 in the same order: imports, CSV, point geometries, a first point map, a second spatial layer, layering, a spatial join, grouped counts, a summary map and export. My study area is central Milan.

The final map uses **WGS 84 / UTM zone 32N (EPSG:32632)**.

The design is a black background with thin street lines, dashed square-density contours and white concentric symbols. The title is playful; the data represent sightings, not a count of individual birds. The final PDF is one 16:9 page. No named class precedent has been supplied, so no attribution is invented.


## 1. Imports and folders 

In [19]:
import numpy as np
import pandas as pd
import geopandas as gpd
import altair as alt
import rasterio
import matplotlib.pyplot as plt
import os
from pathlib import Path

DATA = Path('data/original')
OUTPUT = Path('output')
OUTPUT.mkdir(exist_ok=True)
CRS = 'EPSG:32632'
FONT = 'Roboto'
AUTHOR = 'Michelle Nuccio'
os.listdir('.')

['.DS_Store',
 'Assignement 02.ipynb',
 'environment.yml',
 'output',
 'tutorial-2.ipynb',
 'fonts',
 'data',
 'tutorial-1.ipynb']

### Raster section 
The tutorial reads New York land-cover data. I do not use that raster for Milan: there is no matching local raster for this subject. I continue with the CSV and vector data. The environment already contains the tutorial libraries, so I do not need to run its rasterio installation cell.

## 2. Read the CSV and build points 

In [20]:
df_Pigeons = pd.read_csv(DATA / 'piccioni_milano_inaturalist.csv')
# I check the coordinates before turning rows into points.
for column in ['longitude', 'latitude', 'positional_accuracy_m']:
    df_Pigeons[column] = pd.to_numeric(df_Pigeons[column], errors='coerce')
valid = df_Pigeons.longitude.between(-180, 180) & df_Pigeons.latitude.between(-90, 90)
df_Pigeons = df_Pigeons.loc[valid].copy()
assert df_Pigeons.observation_id.is_unique
df_Pigeons.head()

,observation_id,scientific_name,observed_on,latitude,longitude,positional_accuracy_m,quality_grade,captive_cultivated,geoprivacy,coordinates_obscured,observer_login,license,observation_url
0,410649,Columba livia domestica,2013-09-15,45.464227,9.189271,26.0,research,False,NaN,False,muir,cc-by,http://www.inaturalist.org/observations/410649
1,1431048,Columba livia,2015-04-27,45.500808,9.253814,NaN,casual,False,NaN,False,emilianomori,cc-by-nc,http://www.inaturalist.org/observations/1431048
2,2432873,Columba livia domestica,2015-10-06,45.464000,9.095900,NaN,casual,False,NaN,False,nicolagilio,cc-by-nc,http://www.inaturalist.org/observations/2432873
3,2730571,Columba livia domestica,2016-02-28,45.451190,9.057519,93.0,casual,False,NaN,False,nicolagilio,cc-by-nc,http://www.inaturalist.org/observations/2730571
4,3441741,Columba livia domestica,2016-06-12,45.499309,9.251959,74.0,casual,False,NaN,False,maurizio-valota,cc0,http://www.inaturalist.org/observations/3441741


In [21]:
gdf_Pigeons = gpd.GeoDataFrame(df_Pigeons,
    geometry=gpd.points_from_xy(df_Pigeons.longitude, df_Pigeons.latitude), crs='EPSG:4326')
# The CSV starts in longitude/latitude. I convert it to metres for Milan.
gdf_Pigeons = gdf_Pigeons.to_crs(CRS)
gdf_Pigeons['easting'] = gdf_Pigeons.geometry.x
gdf_Pigeons['northing'] = gdf_Pigeons.geometry.y
gdf_Pigeons.crs

<Projected CRS: EPSG:32632>
Name: WGS 84 / UTM zone 32N
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Between 6°E and 12°E, northern hemisphere between equator and 84°N, onshore and offshore. Algeria. Austria. Cameroon. Denmark. Equatorial Guinea. France. Gabon. Germany. Italy. Libya. Liechtenstein. Monaco. Netherlands. Niger. Nigeria. Norway. Sao Tome and Principe. Svalbard. Sweden. Switzerland. Tunisia. Vatican City State.
- bounds: (6.0, 0.0, 12.0, 84.0)
Coordinate Operation:
- name: UTM zone 32N
- method: Transverse Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [22]:
df_Pigeons['quality_grade'].unique()

<StringArray>
['research', 'casual']
Length: 2, dtype: str

## 3. First point map 

In [23]:
alt.data_transformers.disable_max_rows()
# I use geoshape points with an identity projection: the coordinates are already UTM.
chart_Pigeons = alt.Chart(gdf_Pigeons).mark_geoshape(
    fill='white', stroke='black', strokeWidth=0.5
).encode(tooltip=['observation_id:N', 'observed_on:N', 'quality_grade:N']).project(
    type='identity', reflectY=True
).properties(width=700, height=700)
chart_Pigeons.properties(background='black')

alt.Chart(...)

## 4. Read the second spatial layer and combine 

In [24]:
gdf_Toponyms = gpd.read_file(DATA / 'milano_centroidi_toponimi.geojson').to_crs(CRS)
# These are square centroids, not polygons like the tutorial's city blocks.
square_types = ['piazza', 'piazzale', 'piazzetta']
selected = gdf_Toponyms.desc_estesa.str.strip().str.casefold().isin(square_types)
gdf_Squares = gdf_Toponyms.loc[selected, ['id_integer', 'denominazione', 'desc_estesa', 'geometry']].copy()
assert gdf_Squares.id_integer.is_unique
gdf_Squares.head()

,id_integer,denominazione,desc_estesa,geometry
3,182354,Piazza CASTELLO,Piazza,POINT (514167.378 5035074.448)
26,182377,Piazza EINAUDI LUIGI,Piazza,POINT (515269.252 5036719.357)
45,182396,Piazza SIMPLICIANO (SAN),Piazza,POINT (514375.692 5035590.595)
56,182407,Piazza MIRABELLO CARLO,Piazza,POINT (514834.769 5035770.766)
58,182409,Piazza MARCO (SAN),Piazza,POINT (514706.636 5035536.074)


In [25]:
chart_Squares = alt.Chart(gdf_Squares).mark_geoshape(
    fill=None, stroke='#bbbbbb', strokeWidth=0.8
).encode(tooltip=['denominazione:N']).project(type='identity', reflectY=True).properties(width=700, height=700)
chart_Squares.properties(background='black')

alt.Chart(...)

In [26]:
combined_Chart = (chart_Squares + chart_Pigeons).properties(background='black')
combined_Chart

alt.LayerChart(...)

## 5. Spatial join and grouped counts
I need areas for the join, but the squares are only points. For the final proportional symbols I build a 100 × 100 m grid. This replaces the tutorial's block polygons. A sighting belongs to one grid cell, and symbol area will represent the number of sightings in that cell.

The map includes all valid-coordinate observations in the selected view, across all dates. It does not correct positional uncertainty or distinguish repeated sightings of the same bird. Cells at the edge may contain only the observations inside the viewport.

In [27]:
from shapely.geometry import box
# I keep the same central view as the previous design.
WEST, SOUTH, EAST, NORTH = 9.165, 45.448, 9.209, 45.482
viewport_wgs84 = gpd.GeoSeries([box(WEST, SOUTH, EAST, NORTH)], crs='EPSG:4326')
viewport_utm = viewport_wgs84.to_crs(CRS).iloc[0]
inside = df_Pigeons.longitude.between(WEST, EAST) & df_Pigeons.latitude.between(SOUTH, NORTH)
gdf_PigeonsView = gdf_Pigeons.loc[inside].copy()
CELL_M = 100
gdf_PigeonsView['grid_x'] = np.floor(gdf_PigeonsView.geometry.x / CELL_M).astype(int)
gdf_PigeonsView['grid_y'] = np.floor(gdf_PigeonsView.geometry.y / CELL_M).astype(int)
gdf_PigeonsView['cell_id'] = gdf_PigeonsView.grid_x.astype(str) + '_' + gdf_PigeonsView.grid_y.astype(str)
occupied = gdf_PigeonsView[['grid_x', 'grid_y', 'cell_id']].drop_duplicates()
gdf_Cells = gpd.GeoDataFrame(occupied, geometry=[
    box(row.grid_x*CELL_M, row.grid_y*CELL_M, (row.grid_x+1)*CELL_M, (row.grid_y+1)*CELL_M)
    for row in occupied.itertuples()], crs=CRS)
gdf_Cells.head()

,grid_x,grid_y,cell_id,geometry
0,5147,50345,5147_50345,"POLYGON ((514800 5034500, 514800 5034600, 5147..."
9,5150,50345,5150_50345,"POLYGON ((515100 5034500, 515100 5034600, 5150..."
10,5147,50351,5147_50351,"POLYGON ((514800 5035100, 514800 5035200, 5147..."
11,5148,50345,5148_50345,"POLYGON ((514900 5034500, 514900 5034600, 5148..."
12,5145,50343,5145_50343,"POLYGON ((514600 5034300, 514600 5034400, 5145..."


In [28]:
# Like the tutorial, I join polygons to points before grouping.
gdf_joined = gpd.sjoin(gdf_Cells[['cell_id', 'geometry']],
    gdf_PigeonsView.drop(columns='cell_id'), how='right', predicate='intersects')
# A point exactly on a grid edge could touch two cells. Its floor-based ID resolves this.
expected_cell = gdf_joined.grid_x.astype(str) + '_' + gdf_joined.grid_y.astype(str)
gdf_joined = gdf_joined[gdf_joined.cell_id == expected_cell]
assert len(gdf_joined) == len(gdf_PigeonsView)
gdf_joined.head()

,index_left,cell_id,observation_id,scientific_name,observed_on,latitude,longitude,positional_accuracy_m,quality_grade,captive_cultivated,geoprivacy,coordinates_obscured,observer_login,license,observation_url,geometry,easting,northing,grid_x,grid_y
0,0,5147_50345,410649,Columba livia domestica,2013-09-15,45.464227,9.189271,26.0,research,False,NaN,False,muir,cc-by,http://www.inaturalist.org/observations/410649,POINT (514796.498 5034539.661),514796.498362,5.034540e+06,5147,50345
9,9,5150_50345,7176012,Columba livia domestica,2017-07-23,45.464215,9.192931,10.0,research,False,NaN,False,thunefeld,cc-by-nc,https://www.inaturalist.org/observations/7176012,POINT (515082.647 5034538.956),515082.647170,5.034539e+06,5150,50345
10,10,5147_50351,7176023,Columba livia domestica,2017-07-23,45.469454,9.188377,30.0,research,False,NaN,False,thunefeld,cc-by-nc,https://www.inaturalist.org/observations/7176023,POINT (514725.277 5035120.241),514725.276852,5.035120e+06,5147,50351
11,11,5148_50345,7214724,Columba livia domestica,2017-07-26,45.464076,9.189453,10.0,research,False,NaN,False,thunefeld,cc-by-nc,https://www.inaturalist.org/observations/7214724,POINT (514810.729 5034522.87),514810.728541,5.034523e+06,5148,50345
12,12,5145_50343,7215589,Columba livia domestica,2017-07-26,45.462183,9.186461,NaN,research,False,NaN,False,thunefeld,cc-by-nc,https://www.inaturalist.org/observations/7215589,POINT (514577.332 5034312.081),514577.331597,5.034312e+06,5145,50343


In [29]:
df_counts = gdf_joined.groupby('cell_id').agg(
    observation_count=('observation_id', 'count'),
    mean_easting=('easting', 'mean'), mean_northing=('northing', 'mean')
).reset_index()
gdf_PigeonCells = gdf_Cells.merge(df_counts, on='cell_id', how='left')
gdf_PigeonCells['observation_count'] = gdf_PigeonCells.observation_count.fillna(0).astype(int)
assert gdf_PigeonCells.observation_count.sum() == len(gdf_PigeonsView)
gdf_PigeonCells.sort_values('observation_count', ascending=False).head(10)

,grid_x,grid_y,cell_id,geometry,observation_count,mean_easting,mean_northing
3,5148,50345,5148_50345,"POLYGON ((514900 5034500, 514900 5034600, 5148...",80,514836.630751,5.034544e+06
0,5147,50345,5147_50345,"POLYGON ((514800 5034500, 514800 5034600, 5147...",16,514793.526480,5.034545e+06
18,5138,50355,5138_50355,"POLYGON ((513900 5035500, 513900 5035600, 5138...",10,513830.148397,5.035560e+06
9,5141,50352,5141_50352,"POLYGON ((514200 5035200, 514200 5035300, 5141...",8,514140.267663,5.035242e+06
10,5139,50352,5139_50352,"POLYGON ((514000 5035200, 514000 5035300, 5139...",8,513961.707058,5.035281e+06
12,5141,50351,5141_50351,"POLYGON ((514200 5035100, 514200 5035200, 5141...",8,514125.077504,5.035124e+06
1,5150,50345,5150_50345,"POLYGON ((515100 5034500, 515100 5034600, 5150...",7,515036.833330,5.034547e+06
30,5149,50345,5149_50345,"POLYGON ((515000 5034500, 515000 5034600, 5149...",7,514936.039455,5.034560e+06
22,5137,50354,5137_50354,"POLYGON ((513800 5035400, 513800 5035500, 5137...",7,513771.660483,5.035455e+06
16,5148,50344,5148_50344,"POLYGON ((514900 5034400, 514900 5034500, 5148...",5,514850.087099,5.034490e+06


### Changing the summary measure
The tutorial calculates average tree diameter. There is no comparable measurement in the pigeon CSV, so I use observation counts. I place one symbol at the mean position of the sightings in each occupied cell.

In [30]:
gdf_Symbols = gpd.GeoDataFrame(df_counts,
    geometry=gpd.points_from_xy(df_counts.mean_easting, df_counts.mean_northing), crs=CRS)
chart_Counts = alt.Chart(gdf_PigeonCells).mark_geoshape(stroke='white', strokeWidth=0.4).encode(
    color=alt.Color('observation_count:Q', scale=alt.Scale(scheme='greys')),
    tooltip=['cell_id:N', 'observation_count:Q']
).project(type='identity', reflectY=True).properties(width=700, height=700)
chart_Counts

alt.Chart(...)

## 6. Additional layers for this design
The remaining cells go beyond the tutorial: square-density contours, street lines, numbered centroids and a fixed-page layout. I keep them after the main join-and-summary sequence so that the tutorial steps are still easy to follow.

In [31]:
import contourpy
from shapely.geometry import LineString
# I smooth the square points to get dashed density contours instead of separate circles.
SIGMA_M, GRID_STEP_M = 220, 50
xmin, ymin, xmax, ymax = gdf_Squares.total_bounds
x = np.arange(xmin-4*SIGMA_M, xmax+4*SIGMA_M, GRID_STEP_M)
y = np.arange(ymin-4*SIGMA_M, ymax+4*SIGMA_M, GRID_STEP_M)
xx, yy = np.meshgrid(x, y)
density = np.zeros_like(xx, dtype=float)
for point in gdf_Squares.geometry:
    density += np.exp(-((xx-point.x)**2+(yy-point.y)**2)/(2*SIGMA_M**2))
density /= density.max()
contours = contourpy.contour_generator(x=x, y=y, z=density, line_type='Separate')
records = [{'density_level': level, 'geometry': LineString(coords)}
    for level in [0.12, 0.25, 0.40, 0.60, 0.80]
    for coords in contours.lines(level) if len(coords) >= 2]
gdf_Contours = gpd.GeoDataFrame(records, crs=CRS)
# These curves show the concentration of square centroids, not square boundaries.
gdf_ContoursView = gpd.clip(gdf_Contours, viewport_utm)
gdf_ContoursView.head()

,density_level,geometry
12,0.12,"LINESTRING (512896.816 5035639.198, 512922.283..."
4,0.12,"LINESTRING (516342.958 5032958.409, 516327.744..."
37,0.25,"LINESTRING (516334.949 5036038.655, 516322.283..."
49,0.60,"LINESTRING (514616.789 5033927.467, 514622.283..."
50,0.80,"LINESTRING (514475.382 5034177.467, 514522.283..."


In [32]:
gdf_Streets = gpd.read_file(DATA / 'milano_strade_dbt2020.geojson').to_crs(CRS)
gdf_Streets['geometry'] = gdf_Streets.geometry.simplify(3, preserve_topology=True)
gdf_StreetsView = gpd.clip(gdf_Streets, viewport_utm)
squares_ll = gdf_Squares.to_crs('EPSG:4326')
visible = squares_ll.geometry.x.between(WEST, EAST) & squares_ll.geometry.y.between(SOUTH, NORTH)
gdf_SquaresView = gdf_Squares.loc[visible].sort_values(['denominazione', 'id_integer']).reset_index(drop=True)
gdf_SquaresView['square_id'] = [f'{i:03d}' for i in range(1, len(gdf_SquaresView)+1)]
print(f'{len(gdf_PigeonsView)} sightings, {len(gdf_Symbols)} occupied cells, {len(gdf_SquaresView)} visible squares')

279 sightings, 93 occupied cells, 103 visible squares


## 7. Final composition
I use a fixed 2400 × 1350 page, which is exactly 16:9. 

The scale bar uses that same metres-to-page conversion. The north arrow follows true north at the centre of the view, calculated by projecting a short northward step. The index uses three columns to keep all 103 square names readable on one landscape page.

In [33]:
import vl_convert as vlc
from shapely.ops import transform
from shapely.geometry import mapping, MultiLineString
from pyproj import Transformer
vlc.register_font_directory(str(Path('fonts').resolve()))
PAGE_W, PAGE_H = 2400, 1350
MAP_X, MAP_Y, MAP_W, MAP_H = 55, 195, 920, 1010
xmin, ymin, xmax, ymax = viewport_utm.bounds
METRES_TO_PAGE = min(MAP_W/(xmax-xmin), MAP_H/(ymax-ymin))
map_left = MAP_X + (MAP_W-(xmax-xmin)*METRES_TO_PAGE)/2
map_top = MAP_Y + (MAP_H-(ymax-ymin)*METRES_TO_PAGE)/2

def page_xy(easting, northing):
    return map_left+(np.asarray(easting)-xmin)*METRES_TO_PAGE, map_top+(ymax-np.asarray(northing))*METRES_TO_PAGE

def page_geometry(geometry):
    return transform(lambda x,y,z=None: page_xy(x,y), geometry)

# I can use the already projected linework without asking Altair to project it again.
def line_layer(gdf, colour, width, dash=None, opacity=1):
    parts = []
    for geom in gdf.geometry:
        if geom.geom_type == 'LineString': parts.append(geom)
        elif geom.geom_type == 'MultiLineString': parts.extend(geom.geoms)
    feature = {'type':'Feature', 'geometry':mapping(page_geometry(MultiLineString(parts))), 'properties':{}}
    mark = dict(filled=False, fill=None, stroke=colour, strokeWidth=width, opacity=opacity)
    if dash: mark['strokeDash']=dash
    return alt.Chart(alt.Data(values=[feature])).mark_geoshape(**mark).project(type='identity',scale=1,translate=[0,0])

X = alt.X('x:Q', scale=alt.Scale(domain=[0,PAGE_W],nice=False), axis=None)
Y = alt.Y('y:Q', scale=alt.Scale(domain=[PAGE_H,0],nice=False), axis=None)

def text_layer(rows, size, colour='white', align='left'):
    return alt.Chart(pd.DataFrame(rows)).mark_text(font=FONT,fontWeight=300,fontSize=size,
        color=colour,align=align,baseline='middle').encode(x=X,y=Y,text='text:N')

chart_Streets = line_layer(gdf_StreetsView,'#6c6c6c',0.48,opacity=0.75)
chart_Density = line_layer(gdf_ContoursView,'#dddddd',0.95,dash=[3,3],opacity=0.9)
# One symbol per occupied cell; all ring areas keep the same proportions.
symbols = pd.DataFrame(gdf_Symbols.drop(columns='geometry'))
symbols['x'],symbols['y'] = page_xy(gdf_Symbols.geometry.x,gdf_Symbols.geometry.y)
max_count = int(symbols.observation_count.max())
MAX_AREA = 2500
symbols['inner_area'] = symbols.observation_count/max_count*MAX_AREA*0.32
symbols['dot_area'] = symbols.observation_count/max_count*MAX_AREA*0.055
size_scale = alt.Scale(domain=[0,max_count],range=[0,MAX_AREA])
base = alt.Chart(symbols).encode(x=X,y=Y,tooltip=[alt.Tooltip('observation_count:Q',title='Pigeon sightings'),alt.Tooltip('cell_id:N',title='100 m grid cell')])
chart_Outer = base.mark_circle(fill='black',stroke='white',strokeWidth=1.6,opacity=1).encode(size=alt.Size('observation_count:Q',scale=size_scale,legend=None))
chart_Inner = base.mark_circle(fill='black',stroke='white',strokeWidth=1,opacity=1).encode(size=alt.Size('inner_area:Q',scale=None,legend=None))
chart_Dots = base.mark_circle(color='white',opacity=1).encode(size=alt.Size('dot_area:Q',scale=None,legend=None))

# I keep the plus sign on the true centroid and place its number nearby.
centroids = pd.DataFrame(gdf_SquaresView.drop(columns='geometry'))
centroids['x'],centroids['y'] = page_xy(gdf_SquaresView.geometry.x,gdf_SquaresView.geometry.y)
number_rows, occupied_boxes = [], []
for row in centroids.itertuples():
    options=[]
    for dx,dy in [(18,0),(-18,0),(18,-12),(18,12),(-18,-12),(-18,12),(0,-19),(0,19),(30,-22),(-30,22)]:
        a,b=row.x+dx,row.y+dy
        box_label=(a-11,b-6,a+11,b+6)
        overlap=sum(max(0,min(box_label[2],q[2])-max(box_label[0],q[0]))*max(0,min(box_label[3],q[3])-max(box_label[1],q[1])) for q in occupied_boxes)
        outside=100000 if not (MAP_X+12<a<MAP_X+MAP_W-12 and MAP_Y+8<b<MAP_Y+MAP_H-8) else 0
        options.append((overlap*100+dx*dx+dy*dy+outside,a,b,box_label))
    _,a,b,box_label=min(options,key=lambda item:item[0]);occupied_boxes.append(box_label)
    number_rows.append({'x':a,'y':b,'text':row.square_id})
chart_Plus = alt.Chart(centroids).mark_text(text='+',font=FONT,fontWeight=300,fontSize=17,color='white').encode(x=X,y=Y,
    tooltip=[alt.Tooltip('square_id:N',title='Square ID'),alt.Tooltip('denominazione:N',title='Square')])
chart_Numbers = text_layer(number_rows,10,'#dddddd','center')

# Three vertical columns fit the full index on a landscape page.
index_rows=[]
for i,row in enumerate(gdf_SquaresView.itertuples()):
    col,position=divmod(i,35)
    index_rows.append({'x':1040+col*430,'y':232+position*22,
                       'text':'+'+row.square_id+'  '+row.denominazione.title()})
chart_Index = text_layer(index_rows,14,'#eeeeee')
chart_Title = text_layer([{'x':55,'y':62,'text':'PIGEONS LOVE PIAZZA DUOMO'}],53)
chart_Subtitle = text_layer([
    {'x':57,'y':117,'text':"Meet Milan's newest tourists: here for Piazza Duomo,"},
    {'x':57,'y':146,'text':'probably staying for aperitivo in Sempione.'}],21,'#cccccc')
chart_Headers = text_layer([{'x':1040,'y':190,'text':'THE PIAZZAS'}],23)
chart_Author = text_layer([{'x':2345,'y':62,'text':AUTHOR},{'x':2345,'y':93,'text':'WGS 84 / UTM zone 32N - EPSG:32632'}],17,'#dddddd','right')

# I derive the scale bar from the same conversion used for every map coordinate.
scale_start_x, scale_y = 75, 1223
scale_rows=[{'x':scale_start_x,'y':scale_y,'group':'scale'},
            {'x':scale_start_x+500*METRES_TO_PAGE,'y':scale_y,'group':'scale'}]
for distance in [0,250,500]:
    sx=scale_start_x+distance*METRES_TO_PAGE
    scale_rows += [{'x':sx,'y':scale_y-5,'group':str(distance)}, {'x':sx,'y':scale_y+5,'group':str(distance)}]
chart_Scale = alt.Chart(pd.DataFrame(scale_rows)).mark_line(stroke='white',strokeWidth=1.3).encode(x=X,y=Y,detail='group:N',order=alt.Order('y:Q'))
chart_ScaleLabels = text_layer([{'x':scale_start_x+d*METRES_TO_PAGE,'y':1241,'text':str(d)+(' m' if d==500 else '')} for d in [0,250,500]],12,'white','center')

# A short geographic northward step gives the true-north direction in the UTM map.
project = Transformer.from_crs('EPSG:4326',CRS,always_xy=True)
lon0,lat0=(WEST+EAST)/2,(SOUTH+NORTH)/2
ex,ny=project.transform(lon0,lat0);ex2,ny2=project.transform(lon0,lat0+0.001)
dx,dy=ex2-ex,-(ny2-ny);norm=np.hypot(dx,dy);dx,dy=dx/norm,dy/norm
ax,ay=945,168;tx,ty=ax+dx*35,ay+dy*35
arrow=[{'x':ax,'y':ay,'group':'shaft'},{'x':tx,'y':ty,'group':'shaft'},
       {'x':tx-dx*10-dy*4,'y':ty-dy*10+dx*4,'group':'head'},
       {'x':tx,'y':ty,'group':'head'},
       {'x':tx-dx*10+dy*4,'y':ty-dy*10-dx*4,'group':'head'}]
for i,r in enumerate(arrow):r['order']=i
chart_North = alt.Chart(pd.DataFrame(arrow)).mark_line(stroke='white',strokeWidth=1.4).encode(x=X,y=Y,detail='group:N',order='order:Q')
chart_N = text_layer([{'x':tx,'y':ty-16,'text':'N'}],16,'white','center')

# This legend uses explicit samples rather than several overlapping automatic legends.
legend_counts=[1,5,10,25,50]
legend=pd.DataFrame({'observation_count':legend_counts,'x':[1080+i*75 for i in range(5)],'y':1105})
chart_LegendCircles=alt.Chart(legend).mark_circle(fill='black',stroke='white',strokeWidth=1.3,opacity=1).encode(x=X,y=Y,size=alt.Size('observation_count:Q',scale=size_scale,legend=None))
chart_LegendNumbers=text_layer([{'x':1080+i*75,'y':1150,'text':str(v)} for i,v in enumerate(legend_counts)],13,'white','center')
chart_LegendDashed=alt.Chart(pd.DataFrame({'x':[1650,1850],'y':[1105,1105]})).mark_line(stroke='white',strokeDash=[3,3],strokeWidth=1).encode(x=X,y=Y)
chart_LegendPlus=text_layer([{'x':2170,'y':1105,'text':'+001'}],20,'white','center')
chart_LegendLabels=text_layer([{'x':1230,'y':1182,'text':'Pigeon sightings / 100 x 100 m'},
 {'x':1750,'y':1150,'text':'Square density'}, {'x':2170,'y':1150,'text':'Squares'}],16,'#dddddd','center')

# Citations are short on the page and fully linked in the HTML and notebook.
source_rows=[
 {'x':1040,'y':1233,'text':'Sightings, not individual birds. 279 records in 93 occupied cells; all recorded dates.', 'url':'https://api.inaturalist.org/v1/observations?taxon_id=3017&place_id=191146&geo=true'},
 {'x':55,'y':1280,'text':'Pigeons: iNaturalist API, Columba livia, Milan (place 191146; taxon 3017). Retrieved 24 Sep 2026. Record-specific licences.', 'url':'https://api.inaturalist.org/v1/observations?taxon_id=3017&place_id=191146&geo=true'},
 {'x':55,'y':1304,'text':'Squares: City of Milan, Toponomastica / Centroidi Vie, layer 0. Retrieved 24 Sep 2026. CC BY 4.0. gisportal.comune.milano.it', 'url':'https://gisportal.comune.milano.it/federated/rest/services/Toponomastica/Centroidi_Vie/MapServer/0'},
 {'x':55,'y':1328,'text':'Streets: City of Milan, DBT 2020 / Label strade. CC BY 4.0. dati.comune.milano.it  |  Roboto Light  |  Michelle Nuccio', 'url':'https://dati.comune.milano.it/dataset/6c60b571-5143-4a1b-a125-23b062da474a'}]
chart_Sources=alt.Chart(pd.DataFrame(source_rows)).mark_text(font=FONT,fontWeight=300,fontSize=13,color='#bbbbbb',align='left').encode(x=X,y=Y,text='text:N',href='url:N')
Final_Chart = alt.layer(chart_Streets,chart_Density,chart_Outer,chart_Inner,chart_Dots,
 chart_Plus,chart_Numbers,chart_Index,chart_Title,chart_Subtitle,chart_Headers,chart_Author,
 chart_Scale,chart_ScaleLabels,chart_North,chart_N,chart_LegendCircles,chart_LegendNumbers,
 chart_LegendDashed,chart_LegendPlus,chart_LegendLabels,chart_Sources
).resolve_scale(size='independent').properties(width=PAGE_W,height=PAGE_H,padding=0,
 autosize=alt.AutoSizeParams(type='none',contains='padding'),background='black').configure_view(stroke=None).configure(font=FONT)
Final_Chart


alt.LayerChart(...)

## 8. Export

In [34]:
import base64
Final_Chart.save(OUTPUT / 'Assignement_02_Milano.pdf')
Final_Chart.save(OUTPUT / 'Assignement_02_Milano.png', scale_factor=2)
Final_Chart.save(OUTPUT / 'Assignement_02_Milano.html', inline=True)
# I embed the font so the browser and PDF use the same light typeface.
html_path = OUTPUT / 'Assignement_02_Milano.html'
font_data = base64.b64encode(Path('fonts/Roboto-Light.ttf').read_bytes()).decode('ascii')
font_css = '<style>@font-face{font-family:Roboto;font-weight:300;src:url(data:font/ttf;base64,' + font_data + ') format("truetype");}body{background:black;color:white;font-family:Roboto;font-weight:300;}</style>'
html = html_path.read_text().replace('</head>', font_css+'</head>')
html = html.replace("vegaEmbed('#vis', spec, embedOpt)", "document.fonts.load('300 16px Roboto').then(() => vegaEmbed('#vis', spec, embedOpt))")
html_path.write_text(html)
gdf_Symbols.drop(columns='geometry').to_csv(DATA / 'Assignement_02_piccioni_aree_100m.csv', index=False)
gdf_SquaresView[['square_id', 'denominazione']].to_csv(DATA / 'Assignement_02_square_index.csv', index=False)
print('Saved maps in output/ and CSV tables in data/original/.')

Saved maps in output/ and CSV tables in data/original/.


## Sources and design notes
- **Pigeon observations:** iNaturalist public API, *Columba livia* and subordinate taxa, Milan place 191146, taxon 3017. Retrieved 24 September 2026. [Query](https://api.inaturalist.org/v1/observations?taxon_id=3017&place_id=191146&geo=true). Licences vary by observation and are recorded in the CSV; missing values are not an open licence.
- **Square centroids:** Municipality of Milan, *Toponomastica / Centroidi Vie*, layer 0. Retrieved 24 September 2026. [Source service](https://gisportal.comune.milano.it/federated/rest/services/Toponomastica/Centroidi_Vie/MapServer/0). CC BY 4.0. Piazza, Piazzale and Piazzetta are selected from the complete toponym dataset.
- **Street context:** Municipality of Milan, DBT 2020, *Label strade*. [Dataset](https://dati.comune.milano.it/dataset/6c60b571-5143-4a1b-a125-23b062da474a). CC BY 4.0.
- **Font:** Roboto Light, distributed with its SIL Open Font License in `fonts/OFL.txt`.

The final title is playful and is not a statistical claim about pigeon preferences. A named classroom visual precedent has been omitted as requested; that specific assignment requirement is therefore still outstanding.